In [ ]:
# split the big file to a single small file

input_file = r"C:\Users\kanna\Desktop\GUVI\5.Project\Final Project\simplified-nq-train.jsonl"
output_file = r"C:\Users\kanna\Desktop\GUVI\5.Project\Final Project\split1.jsonl"
lines_to_write = 10000

count = 0

with open(input_file, "r", encoding="utf-8") as f, \
     open(output_file, "w", encoding="utf-8") as out:

    for line in f:
        if count == lines_to_write:
            break
        out.write(line)
        count += 1


In [5]:
# this code will parse the small 10000 line file

from bs4 import BeautifulSoup
import json

def load_nq_dataset(file_path):
    data = []
    with open (file_path, 'r', encoding='utf-8') as f:
        for line in f:
            record = json.loads(line)
            html_content = record.get("document_text", "")

            # Parse HTML and extract text
            soup = BeautifulSoup(html_content, "html.parser")
            plain_text = soup.get_text(separator = " ", strip = True)
            data.append(plain_text)
    return data

In [ ]:
# This is just to print the output. 

load_nq_dataset(r"C:\Users\kanna\Desktop\GUVI\5.Project\Final Project\split1.jsonl")

In [6]:
# Write the clean output to a different file. 

with open("cleaned_output.txt", "w", encoding="utf-8") as out:
    for text in load_nq_dataset(r"C:\Users\kanna\Desktop\GUVI\5.Project\Final Project\split1.jsonl"):
        out.write(text + "\n")

In [ ]:
# This code will combine the activity of splitting and html parsing - Not using in this pipeline

from bs4 import BeautifulSoup
import json

with open(r"C:\Users\kanna\Desktop\GUVI\5.Project\Final Project\simplified-nq-train.jsonl", "r", encoding="utf-8") as f, \
     open(r"C:\Users\kanna\Desktop\GUVI\5.Project\Final Project\questions_only.jsonl", "w", encoding="utf-8") as out:

    for i, line in enumerate(f):
        if i == 10000:
            break
        record = json.loads(line)
        html_content = record.get("document_text", "")
        # Parse HTML and extract text
        soup = BeautifulSoup(html_content, "html.parser")
        plain_text = soup.get_text(separator = " ", strip = True)
        out.write(json.dumps(plain_text) + "\n")

In [ ]:
# for chunking the data

from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

path = r"C:\Users\kanna\Desktop\GUVI\5.Project\Final Project\cleaned_output.txt"

# Read file manually
with open(path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Convert each line into a Document
from langchain.schema import Document
docs = [Document(page_content=line.strip()) for line in lines]

# Now chunk each question (if needed)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

documents = text_splitter.split_documents(docs)

documents[:5]


c:\Users\kanna\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={}, page_content='Email marketing - Wikipedia Email marketing Jump to : navigation , search ( hide ) This article has multiple issues . Please help improve it or discuss these issues on the talk page . ( Learn how and when to remove these template messages ) This article needs additional citations for verification . Please help improve this article by adding citations to reliable sources . Unsourced material may be challenged and removed . ( September 2014 ) ( Learn how and when to remove this template message ) This article possibly contains original research . Please improve it by verifying the claims made and adding inline citations . Statements consisting only of original research should be removed . ( January 2015 ) ( Learn how and when to remove this template message ) ( Learn how and when to remove this template message ) Part of a series on Internet marketing Search engine optimization Local search engine optimisation Social media marketing Email marketing Re

In [ ]:
# not using this

!pip3 install faiss-cpu

   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ----------------- ---------------------- 8.4/18.9 MB 50.2 MB/s eta 0:00:01
   ---------------------------------------  18.6/18.9 MB 48.6 MB/s eta 0:00:01
   ---------------------------------------- 18.9/18.9 MB 44.4 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Creating the db

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os

# Lighter model option (if needed; original is fine for most cases)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-MiniLM-L3-v2")

# Split into smaller batches
batch_size = 20  # Reduced; test and increase if stable
batches = [documents[i:i + batch_size] for i in range(0, len(documents), batch_size)]

# Persistence directory (offloads to disk)
persist_dir = "./chroma_db"
os.makedirs(persist_dir, exist_ok=True)

# Create Chroma DB incrementally with persistence
db = None
for i, batch in enumerate(batches):
    try:
        if db is None:
            db = Chroma.from_documents(
                batch, 
                embedding_model, 
                persist_directory=persist_dir
            )
        else:
            db.add_documents(batch)
        
        # Persist after each batch to free RAM
        db.persist()
        print(f"Processed batch {i+1}/{len(batches)} ({len(batch)} docs)")
        
    except MemoryError:
        print(f"Memory error at batch {i+1}. Try smaller batch_size.")
        break
    except Exception as e:
        print(f"Error in batch {i+1}: {e}")
        break

if db is not None:
    print(f"DB created successfully with {db._collection.count()} total docs.")
else:
    print("Failed to create DB.")


C:\Users\kanna\AppData\Local\Temp\ipykernel_57520\1688611154.py:30: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


Processed batch 1/24167 (20 docs)
Processed batch 2/24167 (20 docs)
Processed batch 3/24167 (20 docs)
Processed batch 4/24167 (20 docs)
Processed batch 5/24167 (20 docs)
Processed batch 6/24167 (20 docs)
Processed batch 7/24167 (20 docs)
Processed batch 8/24167 (20 docs)
Processed batch 9/24167 (20 docs)
Processed batch 10/24167 (20 docs)
Processed batch 11/24167 (20 docs)
Processed batch 12/24167 (20 docs)
Processed batch 13/24167 (20 docs)
Processed batch 14/24167 (20 docs)
Processed batch 15/24167 (20 docs)
Processed batch 16/24167 (20 docs)
Processed batch 17/24167 (20 docs)
Processed batch 18/24167 (20 docs)
Processed batch 19/24167 (20 docs)
Processed batch 20/24167 (20 docs)
Processed batch 21/24167 (20 docs)
Processed batch 22/24167 (20 docs)
Processed batch 23/24167 (20 docs)
Processed batch 24/24167 (20 docs)
Processed batch 25/24167 (20 docs)
Processed batch 26/24167 (20 docs)
Processed batch 27/24167 (20 docs)
Processed batch 28/24167 (20 docs)
Processed batch 29/24167 (20 

In [ ]:
# Not used in this pipeline - Load the database back

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-MiniLM-L3-v2"
)

db = FAISS.load_local("faiss_index", embedding_model, allow_dangerous_deserialization=True)


In [11]:
# This is for loading the db

from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-MiniLM-L3-v2")

db = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings
)


In [15]:
# This block is used to verify or test a piece of code in the streamlit application. 

def load_embedding_model():
    return HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-MiniLM-L3-v2")

def load_chroma_db(persist_dir="chroma_db"):
    embedding_model = load_embedding_model()
    db = Chroma(persist_directory=persist_dir, embedding_function=embedding_model)
    return db

db = load_chroma_db()
db._collection.count()

262160

In [12]:
# Querying the DB  

import textwrap

query = "Explain self attention"
retrival_docs = db.similarity_search(query) #--> This will fetch embeddings and inturn will fetch the final information.
wrapped = textwrap.fill(retrival_docs[0].page_content, width=40)
print(wrapped)
# print (retrival_docs[0].page_content)

, Focused Attention ( FA ) meditation ,
entails the voluntary focusing of
attention on a chosen object , breathing
, image , or words . The other style ,
Open Monitoring ( OM ) meditation ,
involves non-reactive monitoring of the
content of experience from moment to
moment . Direction of mental attention
... A practitioner can focus intensively
on one particular object ( so - called
concentrative meditation ) , on all
mental events that enter the field of
awareness ( so - called mindfulness
meditation ) , or both specific focal
points and the field of awareness .
Focused attention methods ( edit ) These
include paying attention to the breath ,
to an idea or feeling ( such as mettā (
loving - kindness ) ) , or to a mantra (
such as in transcendental meditation ) ,
and single point meditation . Open
monitoring methods ( edit ) These
include mindfulness , shikantaza and
other awareness states . Practices using
both methods ( edit ) Some practices use
both techniques , including vipassana


In [9]:
# Get top K,  similar docs

import textwrap

query = "Explain self attention"
retrival_docs = db.similarity_search(query, k=2)
for doc in retrival_docs:
    wrap = textwrap.fill(doc.page_content, width = 40)
    print (wrap)
    print ("\n")
    print ("\n")

, Focused Attention ( FA ) meditation ,
entails the voluntary focusing of
attention on a chosen object , breathing
, image , or words . The other style ,
Open Monitoring ( OM ) meditation ,
involves non-reactive monitoring of the
content of experience from moment to
moment . Direction of mental attention
... A practitioner can focus intensively
on one particular object ( so - called
concentrative meditation ) , on all
mental events that enter the field of
awareness ( so - called mindfulness
meditation ) , or both specific focal
points and the field of awareness .
Focused attention methods ( edit ) These
include paying attention to the breath ,
to an idea or feeling ( such as mettā (
loving - kindness ) ) , or to a mantra (
such as in transcendental meditation ) ,
and single point meditation . Open
monitoring methods ( edit ) These
include mindfulness , shikantaza and
other awareness states . Practices using
both methods ( edit ) Some practices use
both techniques , including vipassana


In [10]:
# Follow up question

import textwrap

query = "Explain self attention"
retrival_docs = db.similarity_search(query, k=2)
first_doc = retrival_docs[0].page_content

follow_up_question = r"Explain attention heads based on the content" + first_doc
follow_up_result = db.similarity_search(follow_up_question)
if follow_up_result[0]:
    wrappy = textwrap.fill(follow_up_result[0].page_content, width = 40)
    print(wrappy)
else:
    print("No information present")


, Focused Attention ( FA ) meditation ,
entails the voluntary focusing of
attention on a chosen object , breathing
, image , or words . The other style ,
Open Monitoring ( OM ) meditation ,
involves non-reactive monitoring of the
content of experience from moment to
moment . Direction of mental attention
... A practitioner can focus intensively
on one particular object ( so - called
concentrative meditation ) , on all
mental events that enter the field of
awareness ( so - called mindfulness
meditation ) , or both specific focal
points and the field of awareness .
Focused attention methods ( edit ) These
include paying attention to the breath ,
to an idea or feeling ( such as mettā (
loving - kindness ) ) , or to a mantra (
such as in transcendental meditation ) ,
and single point meditation . Open
monitoring methods ( edit ) These
include mindfulness , shikantaza and
other awareness states . Practices using
both methods ( edit ) Some practices use
both techniques , including vipassana
